# DP2 object and metadetect shear crossmatch (tract 9813)

Contact author: Alex Broughton  
Date:


In [ ]:
! eups list -s | grep lsst_distrib


## Introduction

Load all `object` tables for one tract via the Butler `where` clause, load metadetect `object_shear_all` for the same tract, then crossmatch on sky with `astropy.coordinates` (nearest object per shear row). Butler paths match `_tutorials/access-dp2-metadetect-catalogs.ipynb`.


## 1.0 Set Up


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pyarrow as pa
import h5py

from astropy.coordinates import SkyCoord
from astropy import units
from lsst.daf.butler import Butler

REPO = "/sdf/data/rubin/repo/dp2_prep"
COLLECTION = "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3"
TRACT = 9813  # COSMOS field

butler = Butler(
    REPO,
    collections=[COLLECTION],
    skymap="lsst_cells_v2",
    instrument="LSSTCam",
)


TRACT_SIZE_DEG = 1.6 # deg
TRACT_SIZE_ARCMIN = TRACT_SIZE_DEG * 60 # arcmin


### Object column selection



In [ ]:
columns_to_keep = [
    "u_ra",
    "g_ra",
    "r_ra",
    "i_ra",
    "z_ra",
    "y_ra",
    "u_dec",
    "g_dec",
    "r_dec",
    "i_dec",
    "z_dec",
    "y_dec",
    "u_raErr",
    "g_raErr",
    "r_raErr",
    "i_raErr",
    "z_raErr",
    "y_raErr",
    "u_decErr",
    "g_decErr",
    "r_decErr",
    "i_decErr",
    "z_decErr",
    "y_decErr",
    "u_psfFlux",
    "g_psfFlux",
    "r_psfFlux",
    "i_psfFlux",
    "z_psfFlux",
    "y_psfFlux",
    "u_psfFluxErr",
    "g_psfFluxErr",
    "r_psfFluxErr",
    "i_psfFluxErr",
    "z_psfFluxErr",
    "y_psfFluxErr",
    "u_extendedness",
    "g_extendedness",
    "r_extendedness",
    "i_extendedness",
    "z_extendedness",
    "y_extendedness",
    "u_blendedness",
    "g_blendedness",
    "r_blendedness",
    "i_blendedness",
    "z_blendedness",
    "y_blendedness",
    "u_sersicFlux",
    "g_sersicFlux",
    "r_sersicFlux",
    "i_sersicFlux",
    "z_sersicFlux",
    "y_sersicFlux",
    "u_sersicFluxErr",
    "g_sersicFluxErr",
    "r_sersicFluxErr",
    "i_sersicFluxErr",
    "z_sersicFluxErr",
    "y_sersicFluxErr",
    "u_cModelFlux",
    "g_cModelFlux",
    "r_cModelFlux",
    "i_cModelFlux",
    "z_cModelFlux",
    "y_cModelFlux",
    "u_cModelFluxErr",
    "g_cModelFluxErr",
    "r_cModelFluxErr",
    "i_cModelFluxErr",
    "z_cModelFluxErr",
    "y_cModelFluxErr",
    "u_cModel_flag",
    "g_cModel_flag",
    "r_cModel_flag",
    "i_cModel_flag",
    "z_cModel_flag",
    "y_cModel_flag",
    "u_blendedness_flag",
    "g_blendedness_flag",
    "r_blendedness_flag",
    "i_blendedness_flag",
    "z_blendedness_flag",
    "y_blendedness_flag",
    "u_pixelFlags_bad",
    "g_pixelFlags_bad",
    "r_pixelFlags_bad",
    "i_pixelFlags_bad",
    "z_pixelFlags_bad",
    "y_pixelFlags_bad",
    "u_pixelFlags_clipped",
    "g_pixelFlags_clipped",
    "r_pixelFlags_clipped",
    "i_pixelFlags_clipped",
    "z_pixelFlags_clipped",
    "y_pixelFlags_clipped",
    "u_pixelFlags_clippedCenter",
    "g_pixelFlags_clippedCenter",
    "r_pixelFlags_clippedCenter",
    "i_pixelFlags_clippedCenter",
    "z_pixelFlags_clippedCenter",
    "y_pixelFlags_clippedCenter",
    "u_pixelFlags_cr",
    "g_pixelFlags_cr",
    "r_pixelFlags_cr",
    "i_pixelFlags_cr",
    "z_pixelFlags_cr",
    "y_pixelFlags_cr",
    "u_pixelFlags_crCenter",
    "g_pixelFlags_crCenter",
    "r_pixelFlags_crCenter",
    "i_pixelFlags_crCenter",
    "z_pixelFlags_crCenter",
    "y_pixelFlags_crCenter",
    "u_pixelFlags_edge",
    "g_pixelFlags_edge",
    "r_pixelFlags_edge",
    "i_pixelFlags_edge",
    "z_pixelFlags_edge",
    "y_pixelFlags_edge",
    "u_pixelFlags_inexact_psf",
    "g_pixelFlags_inexact_psf",
    "r_pixelFlags_inexact_psf",
    "i_pixelFlags_inexact_psf",
    "z_pixelFlags_inexact_psf",
    "y_pixelFlags_inexact_psf",
    "u_pixelFlags_inexact_psfCenter",
    "g_pixelFlags_inexact_psfCenter",
    "r_pixelFlags_inexact_psfCenter",
    "i_pixelFlags_inexact_psfCenter",
    "z_pixelFlags_inexact_psfCenter",
    "y_pixelFlags_inexact_psfCenter",
    "u_pixelFlags_interpolated",
    "g_pixelFlags_interpolated",
    "r_pixelFlags_interpolated",
    "i_pixelFlags_interpolated",
    "z_pixelFlags_interpolated",
    "y_pixelFlags_interpolated",
    "u_pixelFlags_interpolatedCenter",
    "g_pixelFlags_interpolatedCenter",
    "r_pixelFlags_interpolatedCenter",
    "i_pixelFlags_interpolatedCenter",
    "z_pixelFlags_interpolatedCenter",
    "y_pixelFlags_interpolatedCenter",
    "u_pixelFlags_nodata",
    "g_pixelFlags_nodata",
    "r_pixelFlags_nodata",
    "i_pixelFlags_nodata",
    "z_pixelFlags_nodata",
    "y_pixelFlags_nodata",
    "u_pixelFlags_offimage",
    "g_pixelFlags_offimage",
    "r_pixelFlags_offimage",
    "i_pixelFlags_offimage",
    "z_pixelFlags_offimage",
    "y_pixelFlags_offimage",
    "u_pixelFlags_saturated",
    "g_pixelFlags_saturated",
    "r_pixelFlags_saturated",
    "i_pixelFlags_saturated",
    "z_pixelFlags_saturated",
    "y_pixelFlags_saturated",
    "u_pixelFlags_saturatedCenter",
    "g_pixelFlags_saturatedCenter",
    "r_pixelFlags_saturatedCenter",
    "i_pixelFlags_saturatedCenter",
    "z_pixelFlags_saturatedCenter",
    "y_pixelFlags_saturatedCenter",
    "u_pixelFlags_sensor_edge",
    "g_pixelFlags_sensor_edge",
    "r_pixelFlags_sensor_edge",
    "i_pixelFlags_sensor_edge",
    "z_pixelFlags_sensor_edge",
    "y_pixelFlags_sensor_edge",
    "u_pixelFlags_sensor_edgeCenter",
    "g_pixelFlags_sensor_edgeCenter",
    "r_pixelFlags_sensor_edgeCenter",
    "i_pixelFlags_sensor_edgeCenter",
    "z_pixelFlags_sensor_edgeCenter",
    "y_pixelFlags_sensor_edgeCenter",
    "u_pixelFlags_suspect",
    "g_pixelFlags_suspect",
    "r_pixelFlags_suspect",
    "i_pixelFlags_suspect",
    "z_pixelFlags_suspect",
    "y_pixelFlags_suspect",
    "u_pixelFlags_suspectCenter",
    "g_pixelFlags_suspectCenter",
    "r_pixelFlags_suspectCenter",
    "i_pixelFlags_suspectCenter",
    "z_pixelFlags_suspectCenter",
    "y_pixelFlags_suspectCenter",
    "u_extendedness_flag",
    "g_extendedness_flag",
    "r_extendedness_flag",
    "i_extendedness_flag",
    "z_extendedness_flag",
    "y_extendedness_flag",
    "u_invalidPsfFlag",
    "g_invalidPsfFlag",
    "r_invalidPsfFlag",
    "i_invalidPsfFlag",
    "z_invalidPsfFlag",
    "y_invalidPsfFlag",
    "u_centroid_flag",
    "g_centroid_flag",
    "r_centroid_flag",
    "i_centroid_flag",
    "z_centroid_flag",
    "y_centroid_flag",
    "parentObjectId",
    "coord_ra",
    "coord_dec",
    "coord_raErr",
    "coord_decErr",
    "coord_flag",
    "refBand",
    "refExtendedness",
    "refSizeExtendedness",
    "footprintArea",
    "ebv",
    "detect_isIsolated",
    "detect_fromBlend",
    "detect_isDeblendedModelSource",
    "deblend_peak_center_x",
    "deblend_peak_center_y",
    "deblend_chi2",
    "deblend_blendId",
    "deblend_blendNChild",
    "deblend_parentNPeaks",
    "deblend_parentNChild",
    "shape_flag",
    "g_epoch",
    "u_epoch",
    "r_epoch",
    "i_epoch",
    "z_epoch",
    "y_epoch",
    "objectId",
    "u_model_extendedness",
    "g_model_extendedness",
    "r_model_extendedness",
    "i_model_extendedness",
    "z_model_extendedness",
    "y_model_extendedness",
    "griz_model_extendedness",
]

print(len(columns_to_keep), "object columns")


## 2.0 Tract catalogs

Query all `object` refs with `where="tract IN (9813)"`, then `butler.get` each ref with `parameters={"columns": columns_to_keep}` so only the selected columns are read. Tables are concatenated incrementally to limit peak memory.

Shear: `object_shear_all` for `TRACT`.


In [ ]:
object_refs = list(
    butler.query_datasets("object", where=f"tract IN ({TRACT})")
)
print(len(object_refs), "object refs")

_get_kw = {"parameters": {"columns": columns_to_keep}}

object_table = None
for i, r in enumerate(object_refs):
    piece = butler.get(r, **_get_kw)
    object_table = piece if object_table is None else pa.concat_tables([object_table, piece])

print("object rows:", len(object_table))

shear_table = butler.get("object_shear_all", dataId={"tract": TRACT})
shear_table = shear_table[shear_table["metaStep"] == "ns"]
print("shear rows:", len(shear_table))


In [ ]:
def nJy_to_abmag(f_njy):
    f_njy = np.asarray(f_njy)
    return 31.4 - 2.5*np.log10(f_njy)

### 2.1 Color-color plot

In [ ]:
u = np.asarray(object_table["u_sersicFlux"], dtype=np.float64)
g = np.asarray(object_table["g_sersicFlux"], dtype=np.float64)
r = np.asarray(object_table["r_sersicFlux"], dtype=np.float64)
i = np.asarray(object_table["i_sersicFlux"], dtype=np.float64)
z = np.asarray(object_table["z_sersicFlux"], dtype=np.float64)

u_err = np.asarray(object_table["u_sersicFluxErr"], dtype=np.float64)
g_err = np.asarray(object_table["g_sersicFluxErr"], dtype=np.float64)
r_err = np.asarray(object_table["r_sersicFluxErr"], dtype=np.float64)
i_err = np.asarray(object_table["i_sersicFluxErr"], dtype=np.float64)
z_err = np.asarray(object_table["z_sersicFluxErr"], dtype=np.float64)

u_snr = u / u_err
g_snr = g / g_err
r_snr = r / r_err
i_snr = i / i_err
z_snr = z / z_err

extendedness = np.asarray(object_table["griz_model_extendedness"], dtype=np.float64)
u_blendedness = np.asarray(object_table["u_blendedness_flag"], dtype=bool)
g_blendedness = np.asarray(object_table["g_blendedness_flag"], dtype=bool)
r_blendedness = np.asarray(object_table["r_blendedness_flag"], dtype=bool)
i_blendedness = np.asarray(object_table["i_blendedness_flag"], dtype=bool)
z_blendedness = np.asarray(object_table["z_blendedness_flag"], dtype=bool)
blended = u_blendedness & g_blendedness & r_blendedness & i_blendedness & z_blendedness

tiny = np.finfo(np.float64).tiny
gi = -2.5 * np.log10(np.clip(g, tiny, None) / np.clip(i, tiny, None))
rz = -2.5 * np.log10(np.clip(r, tiny, None) / np.clip(z, tiny, None))

m = np.isfinite(gi) & np.isfinite(rz) & (extendedness > 0.2) # & ~blended 
m = m & (g > 0) & (r > 0) & (i > 0) & (z > 0)  # & (gi > -1) & (gi < 3.5) & (rz > -1) & (rz < 3.5)
m = m & (g_snr > 3) & (r_snr > 3) & (i_snr > 3) & (z_snr > 3)

plt.figure(figsize=(7, 6))
plt.hexbin(rz[m], gi[m], mincnt=1, cmap="plasma", bins="log", extent=(-2, 5, -2, 5))
plt.ylabel("g - i")
plt.xlabel("r - z")
plt.xlim(-2,5)
plt.ylim(-2,5)
plt.colorbar(label="log10(count)")
plt.tight_layout()

plt.show()

## Galaxy Redshift Evolution

In [ ]:
FILENAME = "../_data/CWWSB_LSST_Roman_colors_YJH.hdf5"

BAND_LIST = [
'u',
'g',
'r',
'i',
'z',
'y',
]

# (Beńitez 2004a)
SED_list = [
    "El_B2004a", 
    "Im_B2004a",
    #"SB2_B2004a",
    #"SB3_B2004a",
    "Sbc_B2004a",
    "Scd_B2004a",
    #"ssp_5Myr_z008",
    #"ssp_25Myr_z008",
]

def get_redshift():
    with h5py.File(FILENAME, "r") as hf:
        redshift = hf["redshift"][:]
    return redshift

def get_color(SED_name, band1_name, band2_name):
    
    band1_index = BAND_LIST.index(band1_name)
    band2_index = BAND_LIST.index(band2_name)

    color_name = f"{band1_name}{band2_name}"
    
    with h5py.File(FILENAME, "r") as hf:
        if (band1_index+1)==band2_index:
            color = hf[f"{SED_name}_{color_name}"][:]
        else:
            color = np.zeros_like(hf["redshift"][:])
            
            band1_name_tmp = band1_name
            while 1:
                band2_name_tmp = BAND_LIST[BAND_LIST.index(band1_name_tmp)+1]
                color_name_tmp = f"{band1_name_tmp}{band2_name_tmp}"
                color_tmp = hf[f"{SED_name}_{color_name_tmp}"][:]
                color += color_tmp
                band1_name_tmp = band2_name_tmp
                if band2_name_tmp == band2_name:
                    break
            
    return color

def make_cc_plot2(ax, SED_name, band1_name, band2_name, band3_name, band4_name, color='C0'):

    alpha = 1
    redshift = get_redshift()
    # step: 0.01
    #print(f"max redshift: {np.max(redshift)}")
    sel = redshift <= 2.5
    
    color1 = get_color(SED_name, band1_name, band2_name)
    color2 = get_color(SED_name, band3_name, band4_name)

    def get_name(s):
        """Return the part of s before the first underscore."""
        return s.split('_', 1)[0]
    
    ax.plot(color1[sel], color2[sel], label=get_name(SED_name), linewidth=1, alpha=alpha)

    # 'o', 's', '^', 'D', 'X', 'v'
    # 'o', '>', 's', '*', 'P', 'X'
    sel = redshift==0
    ax.plot(color1[sel], color2[sel], marker='o', markersize=5,  color=color, alpha=alpha)

    sel = redshift==0.5
    ax.plot(color1[sel], color2[sel], marker='D', markersize=5, color=color, alpha=alpha)

    sel = redshift==1.
    ax.plot(color1[sel], color2[sel], marker='s', markersize=5, color=color, alpha=alpha)

    sel = redshift==1.5
    ax.plot(color1[sel], color2[sel], marker='*', markersize=7, color=color, alpha=alpha)

    sel = redshift==2.0
    ax.plot(color1[sel], color2[sel], marker='+', markersize=7, color=color, alpha=alpha)

    sel = redshift==2.5
    ax.plot(color1[sel], color2[sel], marker='x', markersize=7, color=color, alpha=alpha)
    
    return 0



In [ ]:
def cc_seq_parallel(rz):
    return 2.276 * rz - 0.152

def cc_seq_perp(rz):
    return (1/2.276) * rz - 0.152/(2.276**2)

def rzseq_parallel(z):
    return -0.0248*z + 1.604

domain = np.linspace(-2, 5, 1000)



plt.figure(figsize=(7, 6))
plt.hexbin(gi[m], rz[m], mincnt=1, cmap="viridis", bins="log", extent=(-2, 5, -2, 5))
plt.plot(cc_seq_parallel(domain), domain, "r-")
plt.plot(cc_seq_perp(domain), domain, "r--")
plt.text(3.9, 2, r"CCseq$_{||}$", rotation=25, color="red", fontsize=14)
plt.text(1.35, 4, r"CCseq$_{\perp}$", rotation=70, color="red", fontsize=14)
plt.xlabel("g - i")
plt.ylabel("r - z")
plt.xlim(-2,5)
plt.ylim(-2,5)
plt.colorbar(label="log10(count)")
plt.tight_layout()

plt.show()

In [ ]:
u = np.asarray(object_table["u_sersicFlux"], dtype=np.float64)
g = np.asarray(object_table["g_sersicFlux"], dtype=np.float64)
r = np.asarray(object_table["r_sersicFlux"], dtype=np.float64)
i = np.asarray(object_table["i_sersicFlux"], dtype=np.float64)
z = np.asarray(object_table["z_sersicFlux"], dtype=np.float64)

u_err = np.asarray(object_table["u_sersicFluxErr"], dtype=np.float64)
g_err = np.asarray(object_table["g_sersicFluxErr"], dtype=np.float64)
r_err = np.asarray(object_table["r_sersicFluxErr"], dtype=np.float64)
i_err = np.asarray(object_table["i_sersicFluxErr"], dtype=np.float64)
z_err = np.asarray(object_table["z_sersicFluxErr"], dtype=np.float64)

u_snr = u / u_err
g_snr = g / g_err
r_snr = r / r_err
i_snr = i / i_err
z_snr = z / z_err

extendedness = np.asarray(object_table["griz_model_extendedness"], dtype=np.float64)
u_blendedness = np.asarray(object_table["u_blendedness_flag"], dtype=bool)
g_blendedness = np.asarray(object_table["g_blendedness_flag"], dtype=bool)
r_blendedness = np.asarray(object_table["r_blendedness_flag"], dtype=bool)
i_blendedness = np.asarray(object_table["i_blendedness_flag"], dtype=bool)
z_blendedness = np.asarray(object_table["z_blendedness_flag"], dtype=bool)
blended = u_blendedness & g_blendedness & r_blendedness & i_blendedness & z_blendedness

tiny = np.finfo(np.float64).tiny
gi = -2.5 * np.log10(np.clip(g, tiny, None) / np.clip(i, tiny, None))
rz = -2.5 * np.log10(np.clip(r, tiny, None) / np.clip(z, tiny, None))

m = np.isfinite(gi) & np.isfinite(rz) & (extendedness > 0.2) # & ~blended 
m = m & (g > 0) & (r > 0) & (i > 0) & (z > 0)  # & (gi > -1) & (gi < 3.5) & (rz > -1) & (rz < 3.5)
m = m & (g_snr > 5) & (r_snr > 5) & (i_snr > 5) & (z_snr > 5)

plt.figure(figsize=(7, 6))
plt.hexbin(rz[m], gi[m], mincnt=1, gridsize=150, cmap="binary_r", bins="log", extent=(-2, 5, -2, 5), alpha=0.75, edgecolors="none", vmin=1, vmax=1e3)
plt.colorbar(label=r"$\log_{10}(N)$", shrink=0.919)
# plt.plot(domain, cc_seq_parallel(domain),  "r-")
# plt.plot(domain, cc_seq_perp(domain), "r--")

# Plot galaxy evolution
ax = plt.gca()
band1_name, band2_name, band3_name, band4_name = 'r', 'z', 'g', 'i'

for ind, SED in enumerate(SED_list):
    make_cc_plot2(ax, SED, band1_name, band2_name, band3_name, band4_name, f'C{ind}')


plt.scatter([], [], marker='o', color="k", label=r"$z=0$")
plt.scatter([], [], marker='D', color="k", label=r"$z=0.5$")
plt.scatter([], [], marker='s', color="k", label=r"$z=1.0$")
plt.scatter([], [], marker='*', color="k", label=r"$z=1.5$")
plt.scatter([], [], marker='+', color="k", label=r"$z=2.0$")
plt.scatter([], [], marker='x', color="k", label=r"$z=2.5$")
plt.grid(True, ls=":")


plt.ylabel("g - i")
plt.xlabel("r - z")
plt.xlim(-2,5)
plt.ylim(-2,5)
plt.legend(loc="lower right")
plt.gca().set_aspect(1)
plt.tight_layout()


plt.show()

In [ ]:
u = np.asarray(object_table["u_sersicFlux"], dtype=np.float64)
g = np.asarray(object_table["g_sersicFlux"], dtype=np.float64)
r = np.asarray(object_table["r_sersicFlux"], dtype=np.float64)
i = np.asarray(object_table["i_sersicFlux"], dtype=np.float64)
z = np.asarray(object_table["z_sersicFlux"], dtype=np.float64)

u_err = np.asarray(object_table["u_sersicFluxErr"], dtype=np.float64)
g_err = np.asarray(object_table["g_sersicFluxErr"], dtype=np.float64)
r_err = np.asarray(object_table["r_sersicFluxErr"], dtype=np.float64)
i_err = np.asarray(object_table["i_sersicFluxErr"], dtype=np.float64)
z_err = np.asarray(object_table["z_sersicFluxErr"], dtype=np.float64)

u_snr = u / u_err
g_snr = g / g_err
r_snr = r / r_err
i_snr = i / i_err
z_snr = z / z_err

extendedness = np.asarray(object_table["griz_model_extendedness"], dtype=np.float64)
u_blendedness = np.asarray(object_table["u_blendedness_flag"], dtype=bool)
g_blendedness = np.asarray(object_table["g_blendedness_flag"], dtype=bool)
r_blendedness = np.asarray(object_table["r_blendedness_flag"], dtype=bool)
i_blendedness = np.asarray(object_table["i_blendedness_flag"], dtype=bool)
z_blendedness = np.asarray(object_table["z_blendedness_flag"], dtype=bool)
blended = u_blendedness & g_blendedness & r_blendedness & i_blendedness & z_blendedness

tiny = np.finfo(np.float64).tiny
gi = -2.5 * np.log10(np.clip(g, tiny, None) / np.clip(i, tiny, None))
rz = -2.5 * np.log10(np.clip(r, tiny, None) / np.clip(z, tiny, None))

m = np.isfinite(gi) & np.isfinite(rz) & (extendedness > 0.2) # & ~blended 
m = m & (g > 0) & (r > 0) & (i > 0) & (z > 0)  # & (gi > -1) & (gi < 3.5) & (rz > -1) & (rz < 3.5)
m = m & (g_snr > 5) & (r_snr > 5) & (i_snr > 5) & (z_snr > 5)

plt.figure(figsize=(7, 6))
plt.hexbin(rz[m], gi[m], mincnt=1, gridsize=150, cmap="binary_r", bins="log", extent=(-2, 5, -2, 5), alpha=0.75, edgecolors="none", vmin=1, vmax=1e3)
plt.colorbar(label=r"$\log_{10}(N)$", shrink=0.919)
# plt.plot(domain, cc_seq_parallel(domain),  "r-")
# plt.plot(domain, cc_seq_perp(domain), "r--")

# Plot galaxy evolution
ax = plt.gca()
band1_name, band2_name, band3_name, band4_name = 'r', 'z', 'g', 'i'

# for ind, SED in enumerate(SED_list):
#     make_cc_plot2(ax, SED, band1_name, band2_name, band3_name, band4_name, f'C{ind}')

domain = np.linspace(-2, 5, 1000)
def cc_rseq_parallel1(rz):
    return 2.276 * rz - 0.15

def cc_rseq_parallel2(rz):
    return 2.276 * rz + 0.3

def cc_rseq_lo_rz(rz):
    return [0.1] * len(rz)

plt.plot(domain, cc_rseq_parallel1(domain), linewidth=0.5, color="r")
plt.plot(domain, cc_rseq_parallel2(domain), linewidth=0.5, color="r", linestyle="--")
# plt.plot(domain, cc_rseq_lo_rz(domain), linewidth=0.5, color="r", linestyle="--")
# plt.axvline(0.2, linewidth=0.5, color="r", linestyle="--")
plt.axhline(0.75, linewidth=0.5, color="r", linestyle="--")

plt.scatter([], [], marker='o', color="k", label=r"$z=0$")
plt.scatter([], [], marker='D', color="k", label=r"$z=0.5$")
plt.scatter([], [], marker='s', color="k", label=r"$z=1.0$")
plt.scatter([], [], marker='*', color="k", label=r"$z=1.5$")
plt.scatter([], [], marker='+', color="k", label=r"$z=2.0$")
plt.scatter([], [], marker='x', color="k", label=r"$z=2.5$")
plt.grid(True, ls=":")


plt.ylabel("g - i")
plt.xlabel("r - z")
plt.xlim(-2,5)
plt.ylim(-2,5)
plt.legend(loc="lower right")
plt.gca().set_aspect(1)
plt.tight_layout()


plt.show()

In [ ]:
def selection_function(rz, gi):

    def cc_rseq_parallel1(rz):
        return 2.276 * rz - 0.15

    def cc_rseq_parallel2(rz):
        return 2.276 * rz + 0.3
    
    rseq = (gi > cc_rseq_parallel1(rz))
    # rseq &= (gi < cc_rseq_parallel2(rz))
    # rseq &= (gi > 0.75)

    good = ~rseq
    
    return good

In [ ]:
u = np.asarray(object_table["u_sersicFlux"], dtype=np.float64)
g = np.asarray(object_table["g_sersicFlux"], dtype=np.float64)
r = np.asarray(object_table["r_sersicFlux"], dtype=np.float64)
i = np.asarray(object_table["i_sersicFlux"], dtype=np.float64)
z = np.asarray(object_table["z_sersicFlux"], dtype=np.float64)

u_err = np.asarray(object_table["u_sersicFluxErr"], dtype=np.float64)
g_err = np.asarray(object_table["g_sersicFluxErr"], dtype=np.float64)
r_err = np.asarray(object_table["r_sersicFluxErr"], dtype=np.float64)
i_err = np.asarray(object_table["i_sersicFluxErr"], dtype=np.float64)
z_err = np.asarray(object_table["z_sersicFluxErr"], dtype=np.float64)

u_snr = u / u_err
g_snr = g / g_err
r_snr = r / r_err
i_snr = i / i_err
z_snr = z / z_err

extendedness = np.asarray(object_table["griz_model_extendedness"], dtype=np.float64)
u_blendedness = np.asarray(object_table["u_blendedness_flag"], dtype=bool)
g_blendedness = np.asarray(object_table["g_blendedness_flag"], dtype=bool)
r_blendedness = np.asarray(object_table["r_blendedness_flag"], dtype=bool)
i_blendedness = np.asarray(object_table["i_blendedness_flag"], dtype=bool)
z_blendedness = np.asarray(object_table["z_blendedness_flag"], dtype=bool)
blended = u_blendedness & g_blendedness & r_blendedness & i_blendedness & z_blendedness

tiny = np.finfo(np.float64).tiny
gi = -2.5 * np.log10(np.clip(g, tiny, None) / np.clip(i, tiny, None))
rz = -2.5 * np.log10(np.clip(r, tiny, None) / np.clip(z, tiny, None))

m = np.isfinite(gi) & np.isfinite(rz) & (extendedness > 0.2) # & ~blended 
m = m & (g > 0) & (r > 0) & (i > 0) & (z > 0)  # & (gi > -1) & (gi < 3.5) & (rz > -1) & (rz < 3.5)
m = m & (g_snr > 5) & (r_snr > 5) & (i_snr > 5) & (z_snr > 5)
m = m & selection_function(rz, gi)

plt.figure(figsize=(7, 6))
plt.hexbin(rz[m], gi[m], mincnt=1, gridsize=150, cmap="binary_r", bins="log", extent=(-2, 5, -2, 5), alpha=0.75, edgecolors="none", vmin=1, vmax=1e3)
plt.colorbar(label=r"$\log_{10}(N)$", shrink=0.919)

# Plot galaxy evolution
ax = plt.gca()
band1_name, band2_name, band3_name, band4_name = 'r', 'z', 'g', 'i'

plt.scatter([], [], marker='o', color="k", label=r"$z=0$")
plt.scatter([], [], marker='D', color="k", label=r"$z=0.5$")
plt.scatter([], [], marker='s', color="k", label=r"$z=1.0$")
plt.scatter([], [], marker='*', color="k", label=r"$z=1.5$")
plt.scatter([], [], marker='+', color="k", label=r"$z=2.0$")
plt.scatter([], [], marker='x', color="k", label=r"$z=2.5$")
plt.grid(True, ls=":")


plt.ylabel("g - i")
plt.xlabel("r - z")
plt.xlim(-2,5)
plt.ylim(-2,5)
plt.legend(loc="lower right")
plt.gca().set_aspect(1)
plt.tight_layout()


plt.show()

## 3.0 Crossmatch (object ↔ shear)

Match metadetect shear sources to DRP `object` sources on the sky with `astropy.coordinates`. Keep rows with a nearest neighbor within **0.6 mas** (shear→object match for shear; objects touched by any such match are flagged).

Outputs: `shear_xmatch`, `object_xmatch`, `shear_table_xmatch`, `object_table_xmatch`.

In [ ]:
MATCH_RADIUS = 0.2 * units.arcsecond

obj_coords = SkyCoord(
    ra=np.asarray(object_table["coord_ra"]) * units.deg,
    dec=np.asarray(object_table["coord_dec"]) * units.deg,
)

_shear_names = (
    list(shear_table.schema.names)
    if hasattr(shear_table, "schema")
    else list(shear_table.colnames)
)
if "ra" in _shear_names and "dec" in _shear_names:
    _sh_ra, _sh_dec = "ra", "dec"
elif "coord_ra" in _shear_names and "coord_dec" in _shear_names:
    _sh_ra, _sh_dec = "coord_ra", "coord_dec"
else:
    raise KeyError(f"No sky position columns in shear table: {_shear_names[:20]}")

shear_coords = SkyCoord(
    ra=np.asarray(shear_table[_sh_ra]) * units.deg,
    dec=np.asarray(shear_table[_sh_dec]) * units.deg,
)

# Nearest object per shear row; keep shear with a match inside the radius.
obj_idx, sep, _ = shear_coords.match_to_catalog_sky(obj_coords)
shear_xmatch = sep < MATCH_RADIUS
object_xmatch = np.zeros(len(object_table), dtype=bool)
object_xmatch[obj_idx[shear_xmatch]] = True

if hasattr(shear_table, "filter"):
    shear_table_xmatch = shear_table.filter(pa.array(shear_xmatch))
else:
    shear_table_xmatch = shear_table[shear_xmatch]

object_table_xmatch = object_table[object_xmatch]

print(f"match radius: {MATCH_RADIUS.to(units.mas):.3f}")
print(f"object rows:           {len(object_table):,}")
print(f"shear rows:            {len(shear_table):,}")
print(f"shear with object match: {int(shear_xmatch.sum()):,}")
print(f"objects with shear match: {int(object_xmatch.sum()):,}")
print(f"object_table_xmatch:   {len(object_table_xmatch):,}")
print(f"shear_table_xmatch:    {len(shear_table_xmatch):,}")

In [ ]:
def _table_column_names(tbl):
    if hasattr(tbl, "colnames"):
        return list(tbl.colnames)
    if hasattr(tbl, "schema"):
        return list(tbl.schema.names)
    return list(tbl.columns.keys())

def _take_table_rows(tbl, row_idx):
    row_idx = np.asarray(row_idx, dtype=np.int64)
    if hasattr(tbl, "take"):
        return tbl.take(pa.array(row_idx))
    return tbl[row_idx]


def _fits_compatible_unit(unit):
    """Map LSST/metadetect units to FITS-standard strings (deg, arcsec, arcsec2, ...)."""
    if unit is None or unit == units.dimensionless_unscaled:
        return unit
    if unit.is_equivalent(units.deg):
        return units.deg
    if unit.is_equivalent(units.arcsec):
        return units.arcsec
    if unit.is_equivalent(units.arcsec**2):
        return units.arcsec**2
    return None


def prepare_table_for_fits(table):
    """Copy table with FITS-safe column units and without Butler provenance headers."""
    out = table.copy()
    for name in out.colnames:
        col = out[name]
        unit = col.info.unit
        if unit is None:
            continue
        new_unit = _fits_compatible_unit(unit)
        if new_unit is None:
            out[name] = np.asarray(col.value if hasattr(col, "value") else col)
            continue
        if hasattr(col, "to"):
            out[name] = col.to(new_unit)
        else:
            out[name] = np.asarray(col)
            out[name].info.unit = new_unit
    for key in list(out.meta.keys()):
        if str(key).startswith("LSST."):
            del out.meta[key]
    return out


def crossmatch_shear_to_object(
    shear_table,
    object_table,
    match_radius=0.2 * units.arcsecond,
    *,
    shear_ra_col=None,
    shear_dec_col=None,
    object_ra_col="coord_ra",
    object_dec_col="coord_dec",
):
    """Match shear rows to nearest object rows; return aligned pair tables.
    shear_matched[i] and object_matched[i] are partners.
    """
    sh_names = _table_column_names(shear_table)
    if shear_ra_col is None or shear_dec_col is None:
        if "ra" in sh_names and "dec" in sh_names:
            shear_ra_col, shear_dec_col = "ra", "dec"
        elif "coord_ra" in sh_names and "coord_dec" in sh_names:
            shear_ra_col, shear_dec_col = "coord_ra", "coord_dec"
        else:
            raise KeyError(f"No sky columns in shear table: {sh_names[:20]}")
    shear_coords = SkyCoord(
        ra=np.asarray(shear_table[shear_ra_col]) * units.deg,
        dec=np.asarray(shear_table[shear_dec_col]) * units.deg,
    )
    obj_coords = SkyCoord(
        ra=np.asarray(object_table[object_ra_col]) * units.deg,
        dec=np.asarray(object_table[object_dec_col]) * units.deg,
    )
    object_idx, sep, _ = shear_coords.match_to_catalog_sky(obj_coords)
    keep = sep < match_radius
    shear_idx = np.nonzero(keep)[0]
    matched_object_idx = object_idx[keep]
    shear_matched = _take_table_rows(shear_table, shear_idx)
    object_matched = _take_table_rows(object_table, matched_object_idx)
    return shear_matched, object_matched, sep[keep], matched_object_idx

## 3.1 Apply CC selection function to crossmatched object and shear catalogs

In [ ]:
def get_c(obj_table):
    u = np.asarray(obj_table["u_sersicFlux"], dtype=np.float64)
    g = np.asarray(obj_table["g_sersicFlux"], dtype=np.float64)
    r = np.asarray(obj_table["r_sersicFlux"], dtype=np.float64)
    i = np.asarray(obj_table["i_sersicFlux"], dtype=np.float64)
    z = np.asarray(obj_table["z_sersicFlux"], dtype=np.float64)
    
    u_err = np.asarray(obj_table["u_sersicFluxErr"], dtype=np.float64)
    g_err = np.asarray(obj_table["g_sersicFluxErr"], dtype=np.float64)
    r_err = np.asarray(obj_table["r_sersicFluxErr"], dtype=np.float64)
    i_err = np.asarray(obj_table["i_sersicFluxErr"], dtype=np.float64)
    z_err = np.asarray(obj_table["z_sersicFluxErr"], dtype=np.float64)

    return u, g, r, i, z, u_err, g_err, r_err, i_err, z_err

def get_cc(obj_table):
    u, g, r, i, z, u_err, g_err, r_err, i_err, z_err = get_c(obj_table)
    
    u_snr = u / u_err
    g_snr = g / g_err
    r_snr = r / r_err
    i_snr = i / i_err
    z_snr = z / z_err
    
    tiny = np.finfo(np.float64).tiny
    gi = -2.5 * np.log10(np.clip(g, tiny, None) / np.clip(i, tiny, None))
    rz = -2.5 * np.log10(np.clip(r, tiny, None) / np.clip(z, tiny, None))

    return gi, rz

def get_selection_function(obj_table):
    u, g, r, i, z, u_err, g_err, r_err, i_err, z_err = get_c(obj_table)

    gi, rz = get_cc(obj_table)

    u_snr = u / u_err
    g_snr = g / g_err
    r_snr = r / r_err
    i_snr = i / i_err
    z_snr = z / z_err
    
    extendedness = np.asarray(obj_table["griz_model_extendedness"], dtype=np.float64)
    u_blendedness = np.asarray(obj_table["u_blendedness_flag"], dtype=bool)
    g_blendedness = np.asarray(obj_table["g_blendedness_flag"], dtype=bool)
    r_blendedness = np.asarray(obj_table["r_blendedness_flag"], dtype=bool)
    i_blendedness = np.asarray(obj_table["i_blendedness_flag"], dtype=bool)
    z_blendedness = np.asarray(obj_table["z_blendedness_flag"], dtype=bool)
    blended = u_blendedness & g_blendedness & r_blendedness & i_blendedness & z_blendedness
    
    
    
    m = np.isfinite(gi) & np.isfinite(rz) & (extendedness > 0.2) # & ~blended 
    m = m & (g > 0) & (r > 0) & (i > 0) & (z > 0)  # & (gi > -1) & (gi < 3.5) & (rz > -1) & (rz < 3.5)
    m = m & (g_snr > 5) & (r_snr > 5) & (i_snr > 5) & (z_snr > 5)
    m = m & selection_function(rz, gi)

    return m
    

In [ ]:
shear_table_xmatch, object_table_xmatch, xmatch_sep, xmatch_obj_idx = crossmatch_shear_to_object(
    shear_table,
    object_table,
    match_radius=0.2 * units.arcsecond,
)

In [ ]:
gi, rz = get_cc(object_table_xmatch)
gold = get_selection_function(object_table_xmatch)

print("N_gal / arcmin^2 ~", round(float(np.count_nonzero(gold) / (TRACT_SIZE_ARCMIN **2)), 3))

In [ ]:
plt.figure(figsize=(7, 6))
plt.hexbin(rz[gold], gi[gold], mincnt=1, gridsize=150, cmap="viridis", bins="log", extent=(-2, 5, -2, 5), alpha=0.75, edgecolors="none", vmin=1, vmax=1e3)
plt.colorbar(label=r"$\log_{10}(N)$", shrink=0.919)

plt.ylabel("g - i")
plt.xlabel("r - z")
plt.xlim(-2,5)
plt.ylim(-2,5)
plt.gca().set_aspect(1)
plt.tight_layout()


plt.show()

In [ ]:
def _table_colnames(tbl):
    if hasattr(tbl, "colnames"):
        return list(tbl.colnames)
    if hasattr(tbl, "schema"):
        return list(tbl.schema.names)
    return list(tbl.columns.keys())

def apply_shear_quality_cuts(tbl):
    """Metadetect shear quality mask (aligned with Butler selection in section 3)."""
    names = _table_colnames(tbl)
    n = len(tbl)
    m = np.ones(n, dtype=bool)
    if "metaStep" in names:
        m &= np.asarray(tbl["metaStep"]) == "ns"
    # if "image_flags" in names:
    #     m &= np.asarray(tbl["image_flags"]) == 0
    # if "psfOriginal_flags" in names:
    #     m &= np.asarray(tbl["psfOriginal_flags"]) == 0
    # if "bmask_flags" in names:
    #     m &= np.asarray(tbl["bmask_flags"]) == 0
    # if "ormask_flags" in names:
    #     m &= np.asarray(tbl["ormask_flags"]) == 0
    # if "mfrac" in names:
    #     m &= np.asarray(tbl["mfrac"], dtype=np.float64) < 0.1
    # if "gauss_shape_flags" in names:
    #     m &= np.asarray(tbl["gauss_shape_flags"]) == 0
    # if "gauss_flags" in names:
    #     m &= np.asarray(tbl["gauss_flags"]) == 0
    return tbl[m]

In [ ]:
shear_table_gold = apply_shear_quality_cuts(shear_table_xmatch)

In [ ]:
len(shear_table_gold)

In [ ]:
len(shear_table_gold)/(TRACT_SIZE_ARCMIN**2)

In [ ]:
type(shear_table_xmatch)

In [ ]:
OUT_FITS = "/sdf/home/a/abrought/dp2/dp2-clusters-shear-mapping/_data/shear_table_xmatch_gold.fits"
prepare_table_for_fits(shear_table_gold).write(OUT_FITS, overwrite=True)

In [ ]:
shear_table_gold

In [ ]:
plt.figure()
ax = plt.gca()
ax.invert_xaxis()
ax.hexbin(shear_table_gold['ra'], shear_table_gold['dec'])
ax.set_aspect(1)


In [ ]:
print("Has ra/dec:", "ra" in shear_table_gold.colnames, "dec" in shear_table_gold.colnames)
print("Has x/y:", "x" in shear_table_gold.colnames, "y" in shear_table_gold.colnames)
if "x" in shear_table_gold.colnames:
    x, y = shear_table_gold["x"], shear_table_gold["y"]
    print("x range:", np.nanmin(x), np.nanmax(x), "y range:", np.nanmin(y), np.nanmax(y))
ra, dec = shear_table_gold["ra"], shear_table_gold["dec"]
print("ra range:", np.nanmin(ra), np.nanmax(ra))

In [ ]:
150./(28383.24 -1614.0353)

In [ ]:
shear_table_gold